# Player positioning heatmaps — one game

Implements the [pipeline board](https://www.figma.com/board/FU9lMTi6j4VEAN9qKFPOHR/badminton-analysis-pipeline?node-id=0-1).

Run cells in order using the project's Python 3.12 environment. From the project root, install the existing extras with `uv sync --extra monotrack` followed by `uv pip install rtmlib matplotlib ipykernel`, then `uv pip uninstall onnxruntime` and `uv pip install 'onnxruntime-gpu>=1.21,<1.27'` and select that environment as the notebook kernel.

For each frame: detect a court matching the first frame, retain at most one usable player per court half, requiring at least one visible player. Prune unreliable foot landmarks, and keep frames covered by an offline window where **the shuttle moves OR every observed player moves**. Windows use up to eight consecutive eligible frames, including future observations, and accept their entire interval. Shorter runs use their available observations when at least two exist. Invalid frames split runs, and player movement is never measured across a visibility gap. There is no automatic warm-up discard. Rejected frames retain their frame IDs and missing positions.

Player positions are the midpoint of the available foot centres **after projection into court metres**. Available toes are centred first, then combined with the heel; a single surviving landmark or foot is allowed. Missing players remain NaN. Stance radii remain NaN when only one foot is available. Single-foot fallback has a measured 37 cm median / 71 cm 95th-percentile displacement from the two-foot midpoint on the cached observations; this is not ground-truth error. Identities are near/far court sides for this one game, assumed to have no change of ends. The first frame is assumed reliable. Court and motion checks are the agreed approximate non-rally filter; same-view replays and active pauses can pass.

Heatmaps use a fixed **15 cm radius**, with original frame counts and dwell seconds retained. Neither player gets an independent colour normalization.

Raw model observations and calibration persist under `data/player-position-heatmap/<run-id>/`. Completed chunks are reused automatically; interrupted runs resume at the first missing chunk. Analysis thresholds do not affect the raw run ID. Run all cells again to replay cached observations and regenerate plots without model inference.

Model inference requires an NVIDIA CUDA GPU: YOLO, TrackNet and RTMPose all use `gpu_device`. Decoding, court scoring and file compression still use the CPU. Restart the kernel after changing ONNX Runtime packages. The GPU runtime range matches this project's CUDA 12.8 PyTorch wheels ([ONNX Runtime compatibility](https://onnxruntime.ai/docs/execution-providers/CUDA-ExecutionProvider.html#requirements)).

In [ ]:
from pathlib import Path
import sys
import json
import time
from importlib.metadata import version, PackageNotFoundError
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(project_root / 'src'))
# Refresh local helpers when rerunning this cell in an existing kernel.
from importlib import reload
import badminton_analysis.positions as _positions
reload(_positions)
from badminton_analysis.positions import (
    detect_court, court_agreement, align_court, wholebody_court_positions,
    motion_gate, position_heatmaps, court_model_corners, court_model_segments,
    court_width_m, court_length_m, positions_from_observations,
)
import badminton_analysis.observation_store as _observation_store
reload(_observation_store)
from badminton_analysis.observation_store import ObservationStore, file_fingerprint, settings_id

## Video and thresholds

The default input is the local BWF match, from **6:30 to 33:27** (end exclusive). To try the bundled 16-second clip, use its path with `start_seconds = 0` and `end_seconds = 16`. Set `max_frames = 240` for an initial short run, or leave it `None` for the whole game.

All image thresholds below use the **processing resolution**, which defaults to 960 pixels wide. Court acceptance checks line support at the fixed first-frame court. It does not refit or search for a new layout. These thresholds are starting values to tune empirically. Motion uses stance displacement in metres; the shuttle threshold measures maximum deviation from its mean detected pixel position over the window.

In [ ]:
video_candidates = sorted((project_root / 'test_data').glob('*.webm'))
video_path = video_candidates[0] if video_candidates else project_root / 'test_data' / 'bwf-rally-06m29s-06m45s.mp4'
start_seconds = 6*60 + 30 if video_candidates else 0
end_seconds = 33*60 + 27 if video_candidates else 16
max_frames = None

processing_width = 960
chunk_frames = 96  # Maximum frames per new chunk; existing chunks can have any length.
court_min_score = .45
court_max_lines = 40
foot_min_score = .4
court_margin_m = 1.
motion_window = 8
shuttle_deviation_px = 20.
player_displacement_m = .05
heatmap_radius_m = .15
heatmap_cell_m = .05
raw_data_root = project_root / 'data' / 'player-position-heatmap'
resume_run_id = 'f7d755110dc74ca7f882b074'  # Existing run folder name; None creates/selects a run from current settings.

jump_checkpoint = None  # Optional Path to trained JumpSegmenter weights.
jump_projection_path = None  # Optional .npy 3x4 camera matrix for this raw run and resolution.
jump_threshold = .5
gpu_device = 'cuda:0'
extraction_revision = 'v1'  # Bump when changing inline inference/preprocessing below.
person_checkpoint = project_root / 'yolo26m-pose.pt'
rtmpose_url = ('https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/'
               'rtmpose-l_simcc-ucoco_dw-ucoco_270e-384x288-2438fd99_20230728.zip')
rtmpose_checkpoint = project_root / '.cache' / 'rtmpose' / (Path(rtmpose_url).stem + '.onnx')

## Bounded video decoding

TrackNet receives consecutive eight-frame windows before any filtering. Each chunk includes seven context frames on each side, and only its central frames are retained. Absolute source frame IDs are preserved. Only a chunk of resized images is held in memory; the small coordinate arrays are accumulated for plotting.

Dwell time is accepted-frame count divided by container FPS, assuming constant-rate normal playback. It measures observed time in this video, including any retained replay. It does not fill gaps.

In [ ]:
def video_chunks(capture, first, stop, size, width, core_ranges=None):
    if size < 8 or width < 32 or first < 0 or stop-first < 8:
        raise ValueError('Need chunks >=8, width >=32 and a segment of at least eight frames.')
    ranges = ([(start, min(start+size, stop)) for start in range(first, stop, size)]
              if core_ranges is None else list(core_ranges))
    if (any(not first <= start < end <= stop for start, end in ranges)
            or any(current[0] < previous[1] for previous, current in zip(ranges, ranges[1:]))):
        raise ValueError('core_ranges must be ordered, nonoverlapping frame intervals within the segment.')
    frames = []
    buffer_start = next_frame = None
    for core_start, core_stop in ranges:
        read_start = max(first, core_start-7)
        if buffer_start != read_start:
            capture.set(cv2.CAP_PROP_POS_FRAMES, read_start)
            frames = []
            buffer_start = next_frame = read_start
        read_stop = min(stop, core_stop+7)
        while next_frame < read_stop:
            ok, frame = capture.read()
            if not ok:
                raise ValueError(f'Video decoding failed at source frame {next_frame}; no duration inferred for the gap.')
            height = round(frame.shape[0] * width / frame.shape[1])
            frames.append(cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA))
            next_frame += 1
        core = slice(core_start-buffer_start, core_stop-buffer_start)
        yield np.arange(core_start, core_stop), frames, core
        keep_from = max(first, core_stop-7)
        frames = frames[keep_from-buffer_start:]
        buffer_start = keep_from

## Small checks before inference

These verify perspective projection, low-confidence feet, court orientation, both motion paths, resets at rejected frames, empty heatmaps, dwell counts, and frame/context handling. No models or video downloads are needed for these checks.

In [ ]:
_test_points = np.zeros((1, 133, 2))
_test_points[0, 17:23] = [[1, 2], [1.2, 2], [1.1, 1.6], [2, 2], [2.2, 2], [2.1, 1.6]]
_test_H = np.array([[100., 5., 30.], [2., 50., 40.], [.01, .02, 1.]])
_test_pixels = cv2.perspectiveTransform(_test_points, _test_H)
_test_scores = np.ones((1, 133))
_test_feet, _test_stance = wholebody_court_positions(_test_pixels, _test_scores, _test_H)
np.testing.assert_allclose(_test_stance, [[1.6, 1.8]], atol=1e-8)
_test_scores[0, 19] = 0
np.testing.assert_allclose(wholebody_court_positions(_test_pixels, _test_scores, _test_H)[1], [[1.6, 1.9]], atol=1e-8)
assert np.isnan(wholebody_court_positions(_test_pixels, _test_scores, _test_H, allow_partial=False)[1]).all()
_test_corners = cv2.perspectiveTransform(court_model_corners[None], _test_H)[0]
for _order in [[0, 1, 2, 3], [1, 0, 3, 2], [2, 3, 0, 1], [3, 2, 1, 0]]:
    _aligned, _error = align_court({'corners_px': _test_corners[_order]}, _test_corners)
    assert _error < 1e-5
    np.testing.assert_allclose(cv2.perspectiveTransform(court_model_corners[None], _aligned)[0], _test_corners, atol=1e-4)
_test_pos = np.tile([[2., 2.], [3., 10.]], (16, 1, 1))
_test_shuttle = np.full((16, 2), np.nan)
_test_valid = np.ones(16, bool)
assert not motion_gate(_test_pos, _test_shuttle, _test_valid).any()
_test_shuttle[:, 0] = np.arange(16)*10
_test_shuttle[:, 1] = 100
assert motion_gate(_test_pos, _test_shuttle, _test_valid)[7:].all()
_test_shuttle[:] = np.nan
_test_pos[:, :, 0] += np.arange(16)[:, None]*.02
assert motion_gate(_test_pos, _test_shuttle, _test_valid)[7:].all()
_test_pos[:, 1] = [3., 10.]
assert not motion_gate(_test_pos, _test_shuttle, _test_valid).any()
_test_valid[8] = False
_test_pos[:, 1, 0] += np.arange(16)*.02
_test_keep = motion_gate(_test_pos, _test_shuttle, _test_valid)
assert not _test_keep[8]
_test_maps = position_heatmaps(np.tile([[2., 2.], [3., 10.]], (10, 1, 1)), fps=20.)
np.testing.assert_allclose(_test_maps['frame_counts'].sum(axis=(1, 2)), [10, 10])
np.testing.assert_allclose(_test_maps['dwell_seconds'].max(axis=(1, 2)), [.5, .5])
assert not position_heatmaps(np.full((3, 2, 2), np.nan), 20.)['dwell_seconds'].any()

class _FakeCapture:
    def set(self, prop, value):
        self.index = int(value)
    def read(self):
        image = np.full((32, 32, 3), self.index, np.uint8)
        self.index += 1
        return True, image

_test_ids = []
for _ids, _frames, _core in video_chunks(_FakeCapture(), 10, 43, 8, 32):
    assert len(_frames) >= 8
    assert len(_frames) <= 8+14
    np.testing.assert_array_equal([int(f[0, 0, 0]) for f in _frames[_core]], _ids)
    _test_ids.extend(_ids)
np.testing.assert_array_equal(_test_ids, np.arange(10, 43))
print('Projection, motion, heatmap and chunk checks passed.')

_test_frame = np.full((540, 960, 3), 25, np.uint8)
_test_court_H = np.array([[65., 0., 200.], [0., 25., 80.], [0., 0., 1.]])
for _segment in cv2.perspectiveTransform(court_model_segments, _test_court_H):
    cv2.line(_test_frame, tuple(np.rint(_segment[0]).astype(int)),
             tuple(np.rint(_segment[1]).astype(int)), (255, 255, 255), 3)
_test_agreement = court_agreement(_test_frame, _test_court_H)
assert _test_agreement > .8, _test_agreement
_test_shifted_H = _test_court_H.copy()
_test_shifted_H[:2, 2] += 40
assert court_agreement(_test_frame, _test_shifted_H) < .45
assert court_agreement(np.zeros_like(_test_frame), _test_court_H) == 0
_test_offscreen_H = _test_court_H.copy()
_test_offscreen_H[:2, 2] += 2000
assert court_agreement(_test_frame, _test_offscreen_H) == 0
try:
    court_agreement(_test_frame, np.zeros((3, 3)))
except ValueError:
    pass
else:
    raise AssertionError('A singular homography must be rejected.')
print('Proposed-line checks pass for matching, shifted, blank and off-screen courts.')


## Inspect the first-frame court and open persistent observations

This is the reference view for the entire selected game. Court coordinates start at the **near-left corner**, with Y increasing away from the camera. Subsequent frames are scored against those same projected markings and use this fixed homography. Player labels remain near/far sides. Camera movement is not followed.

The manifest identifies the video by absolute path, byte size and modification time, plus the exact frame range, FPS, processing resolution, context settings, model identities, dependency versions and extractor source hashes. Changed extraction settings create a separate run. Increase `extraction_revision` when editing inline inference code. Video content is not hashed; if replacing a video while preserving its path, size and timestamp, change that revision too.

Calibration (homography **court metres → resized image pixels**, oriented corners, support score and reference image) is committed immediately. Existing calibration loads without decoding frames or searching for the court. Court, foot, motion and heatmap thresholds are applied later and reuse this run.

Set `resume_run_id` to an existing directory name under `raw_data_root` to continue that collection. The video, frame interval, resolution, context settings, calibration algorithm settings and model weights must match. CPU/GPU device, runtime versions and extractor source changes are allowed for explicit resumption. Completed chunks and calibration are reused unchanged; only missing chunks are extracted with the current GPU setup. Set it to `None` to select a run using all current metadata.

`chunk_frames` controls only new work. You can resume with `chunk_frames = 512` while retaining earlier 96-frame files. Saved frame intervals are indexed independently of a fixed grid; gaps are filled using the new maximum size, including shorter tail chunks. Larger chunks can change TrackNet's local median background and use more RAM. Their extraction metadata records the requested size.

In [ ]:
if not video_path.is_file():
    raise FileNotFoundError(video_path)
if not np.isfinite([start_seconds, end_seconds]).all() or not 0 <= start_seconds < end_seconds:
    raise ValueError('Use finite start/end times with 0 <= start < end.')
if max_frames is not None and (not isinstance(max_frames, int) or max_frames < 8):
    raise ValueError('max_frames must be None or an integer >=8.')
if processing_width < 32 or chunk_frames < 8 or court_max_lines < 4:
    raise ValueError('Use processing_width >=32, chunk_frames >=8 and court_max_lines >=4.')
_thresholds = [court_min_score, foot_min_score, court_margin_m,
               shuttle_deviation_px, player_displacement_m]
if not np.isfinite(_thresholds).all() or not (0 <= court_min_score <= 1 and 0 <= foot_min_score <= 1):
    raise ValueError('Use finite thresholds and court/foot scores between 0 and 1.')
if min(shuttle_deviation_px, player_displacement_m) <= 0 or court_margin_m < 0 or motion_window < 2:
    raise ValueError('Use positive motion thresholds, a nonnegative margin, and window >=2.')


capture = cv2.VideoCapture(str(video_path))
try:
    fps = capture.get(cv2.CAP_PROP_FPS)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if not capture.isOpened() or not np.isfinite(fps) or fps <= 0 or total_frames < 8:
        raise ValueError('Video must open with a valid FPS and frame count.')
    source_size = [int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
                   int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))]
finally:
    capture.release()
first_frame_id = int(np.ceil(start_seconds*fps))
stop_frame_id = min(int(np.ceil(end_seconds*fps)), total_frames)
if max_frames is not None:
    stop_frame_id = min(stop_frame_id, first_frame_id+max_frames)
if stop_frame_id-first_frame_id < 8:
    raise ValueError('The selected video segment must contain at least eight frames.')

# Fingerprinting weights needs no model initialization. Fetch RTMPose only if absent.
if not person_checkpoint.is_file():
    raise FileNotFoundError(f'Expected the existing local checkpoint: {person_checkpoint}')
if not rtmpose_checkpoint.is_file():
    from rtmlib.tools.file import download_checkpoint
    rtmpose_checkpoint = Path(download_checkpoint(rtmpose_url, dst_dir=str(rtmpose_checkpoint.parent)))
from badminton_analysis.monotrack.assets import TRACKNET_SHA256

def installed_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None

extraction_metadata = {
    'video': file_fingerprint(video_path), 'fps': fps, 'source_size_px': source_size,
    'first_frame_id': first_frame_id, 'stop_frame_id': stop_frame_id,
    'processing_width': processing_width, 'chunk_frames': chunk_frames,
    'context_frames_each_side': 7, 'court_max_lines': court_max_lines,
    'extraction_revision': extraction_revision,
    'models': {'person': file_fingerprint(person_checkpoint, content=True),
               'rtmpose': file_fingerprint(rtmpose_checkpoint, content=True),
               'rtmpose_url': rtmpose_url, 'rtmpose_input_size': [288, 384],
               'rtmpose_backend': 'onnxruntime', 'rtmpose_device': gpu_device,
               'tracknet_sha256': TRACKNET_SHA256, 'tracknet_batch_size': 4,
               'person_confidence': .25, 'person_imgsz': 960, 'inference_device': gpu_device},
    'extractor_sources': {name: file_fingerprint(project_root / 'src' / 'badminton_analysis' / name, content=True)
                          for name in ['positions.py', 'monotrack/tracking.py', 'monotrack/tracknet_model.py']},
    'versions': {name: installed_version(name) for name in
                 ['numpy', 'opencv-python', 'torch', 'ultralytics', 'rtmlib', 'onnxruntime-gpu', 'pillow']},
}
if resume_run_id is None:
    raw_store = ObservationStore(raw_data_root, extraction_metadata)
else:
    raw_store = ObservationStore.open(raw_data_root / resume_run_id)
    raw_store.validate_resume(extraction_metadata)
started = time.monotonic()
calibration = raw_store.load_calibration()
calibration_cached = calibration is not None
if calibration is None:
    capture = cv2.VideoCapture(str(video_path))
    try:
        _, first_images, _ = next(video_chunks(capture, first_frame_id, stop_frame_id, chunk_frames, processing_width))
        reference_image = first_images[0].copy()
    finally:
        capture.release()
    reference_layout = detect_court(reference_image, max_lines=court_max_lines)
    reference_corners = reference_layout['corners_px'].copy()
    if reference_corners[:2, 1].mean() < reference_corners[2:, 1].mean():
        reference_corners = reference_corners[[3, 2, 1, 0]]
    if reference_corners[0, 0] > reference_corners[1, 0]:
        reference_corners = reference_corners[[1, 0, 3, 2]]
    reference_H, _ = align_court(reference_layout, reference_corners)
    reference_score = reference_layout['score']
    raw_store.save_calibration(reference_image=reference_image, reference_H=reference_H,
                               reference_corners=reference_corners, reference_score=reference_score)
    del first_images
else:
    reference_image = calibration['reference_image_bgr']
    reference_H = calibration['H_court_to_image']
    reference_corners = calibration['reference_corners_px']
    reference_score = float(calibration['reference_score'])
court_seconds = time.monotonic()-started
if reference_score < court_min_score:
    raise ValueError('First-frame court support is below court_min_score; inspect the calibration/thresholds.')
fig, axis = plt.subplots(figsize=(12, 7))
axis.imshow(cv2.cvtColor(reference_image, cv2.COLOR_BGR2RGB))
for segment in court_model_segments:
    pixels = cv2.perspectiveTransform(segment[None], reference_H)[0]
    axis.plot(*pixels.T, color='cyan', linewidth=1)
for name, pixel in zip(['near-left', 'near-right', 'far-right', 'far-left'], reference_corners):
    axis.annotate(name, pixel, color='yellow')
axis.set_title(f"First-frame court: support {reference_score:.3f}; {'cached' if calibration_cached else 'detected'} in {court_seconds:.2f} s")
axis.axis('off')
plt.show()
print(f'{stop_frame_id-first_frame_id:,} frames at {fps:.6f} FPS; source frames [{first_frame_id}, {stop_frame_id}).')
print('Persistent observations:', raw_store.directory)
missing_chunks = raw_store.missing_chunks(chunk_frames)
saved_frames = stop_frame_id-first_frame_id-sum(stop-first for first, stop in missing_chunks)
print(f'Reusing {saved_frames:,} saved frames; {len(missing_chunks)} of {len(raw_store.chunk_bounds(chunk_frames))} chunks need extraction.')

## Load models once for missing raw chunks

YOLO, RTMPose-l WholeBody and TrackNet all run model inference on `gpu_device`. Startup checks require CUDA in PyTorch and ONNX Runtime, verify that the pose session actually uses CUDA, and disable runtime retries on CPU. Frames still use CPU preprocessing and court scoring. TrackNet's weights are verified and loaded once, then reused for every chunk; its context windows and median background remain chunk-local.

Completed raw runs skip model loading entirely. Use `resume_run_id` to reuse earlier CPU observations and continue missing chunks on GPU. Each new chunk references its extraction metadata, including actual device, dependency versions and extractor source hashes. Earlier chunks without an `extraction_id` use the original run manifest as provenance.


In [ ]:
missing_chunks = raw_store.missing_chunks(chunk_frames)
if missing_chunks:
    import torch  # Import first so ONNX Runtime can share PyTorch's CUDA/cuDNN libraries.
    import onnxruntime as ort
    from badminton_analysis.monotrack.assets import fetch_tracknet
    import badminton_analysis.monotrack.tracking as _tracking
    reload(_tracking)

    torch.set_num_threads(4)
    inference_device = _tracking.require_cuda_device(gpu_device)
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        raise RuntimeError(
            "RTMPose requires ONNX Runtime CUDA. Install 'onnxruntime-gpu>=1.21,<1.27' "
            "after removing 'onnxruntime', then restart the kernel.")
    ort.preload_dlls()
    from ultralytics import YOLO
    from rtmlib import RTMPose

    person_detector = YOLO(str(person_checkpoint)).to(str(inference_device))
    wholebody_pose = RTMPose(str(rtmpose_checkpoint), model_input_size=(288, 384),
                            to_openpose=False, backend='onnxruntime', device=str(inference_device))
    if wholebody_pose.session.get_providers()[0] != 'CUDAExecutionProvider':
        raise RuntimeError('RTMPose could not initialize CUDA; refusing CPU inference. Check CUDA/cuDNN libraries.')
    wholebody_pose.session.disable_fallback()
    tracknet_checkpoint = fetch_tracknet(project_root / '.cache' / 'monotrack-assets')
    shuttle_tracker = _tracking.TrackNetTracker(tracknet_checkpoint, inference_device)
    print('YOLO / RTMPose / TrackNet ready on:', inference_device)
    print('RTMPose providers:', wholebody_pose.session.get_providers())
else:
    print('All raw chunks cached; model loading skipped.')

## Extract and checkpoint raw observations, then replay analysis

Every source frame gets a fixed-court support score, person boxes with confidence, **all** WholeBody poses with landmark scores, and the TrackNet shuttle pixel position plus detection flag. Poses are extracted even on frames that fail the current court or player checks, so changing those thresholds can recover observations later. Person detections still use the extraction confidence floor `.25`; changing that floor requires a new extraction revision. No filtering, interpolation or clipping changes these raw arrays.

New compressed NPZs store scalar `chunk_frame_start`, `chunk_frame_stop` (exclusive) and `chunk_frame_count` so each binary file describes its actual interval. Legacy files infer these fields from their saved frame IDs and filename without being rewritten. Each NPZ uses `frame_ids`, absolute `timestamps_seconds`, `court_scores`, `shuttle_pixels`, `shuttle_detected`, `boxes_xyxy`, `box_scores`, `keypoints_px` and `keypoint_scores`. Variable person counts use `pose_offsets`: frame *i* owns the person rows `[pose_offsets[i]:pose_offsets[i+1]]`. Pixels use the resized processing image; the homography and reference image live in `calibration.npz`. Missing shuttle positions remain NaN, and zero-person frames have empty slices. No pickle is needed.

Chunks are written atomically as soon as inference completes, before analysis. On interruption, rerun cells: only missing chunks are decoded and inferred, with seven-frame context and a TrackNet median background local to each new interval. New files also record `context_frame_start` and `context_frame_stop`. Chunk sizes may vary within a run; the replay loader orders actual intervals and rejects overlaps or missing frames. At most the in-progress chunk is lost. After extraction, projection, player selection and motion filtering run entirely from disk. The motion gate runs across chunk boundaries.

Motion classification is offline: every frame in a window with confirmed movement is retained, including the leading frames. Windows never cross invalid observations. Valid runs of two to seven frames use a shorter window; a single isolated observation cannot establish movement. No frame is rejected solely to warm up a trailing window.

In [ ]:
started = time.monotonic()
missing_ranges = raw_store.missing_chunks(chunk_frames)
if missing_ranges:
    extraction_id = raw_store.record_extraction(extraction_metadata)
    capture = cv2.VideoCapture(str(video_path))
    try:
        with tqdm(total=sum(stop-first for first, stop in missing_ranges), desc='Extract raw frames') as progress:
            for ids, images, core in video_chunks(capture, first_frame_id, stop_frame_id,
                                                  chunk_frames, processing_width, core_ranges=missing_ranges):
                # Existing chunks are untouched; this missing interval gets its own background.
                shuttle_result = np.asarray(shuttle_tracker(images, batch_size=4), float)[core]
                court_scores_chunk = []
                boxes_chunk, box_scores_chunk, points_chunk, scores_chunk = [], [], [], []
                pose_offsets = [0]
                for frame_id, image in zip(ids, images[core]):
                    court_scores_chunk.append(reference_score if frame_id == first_frame_id
                                              else court_agreement(image, reference_H))
                    result = person_detector.predict(image, classes=[0], conf=.25, imgsz=960,
                                                     device=str(inference_device), verbose=False)[0]
                    boxes = result.boxes.xyxy.cpu().numpy()
                    box_scores = result.boxes.conf.cpu().numpy()
                    if len(boxes):
                        keypoints, scores = wholebody_pose(image, bboxes=boxes)
                    else:
                        keypoints, scores = np.empty((0, 133, 2)), np.empty((0, 133))
                    boxes_chunk.append(boxes)
                    box_scores_chunk.append(box_scores)
                    points_chunk.append(keypoints)
                    scores_chunk.append(scores)
                    pose_offsets.append(pose_offsets[-1]+len(boxes))
                    progress.update(1)
                raw_store.save_chunk(int(ids[0]), int(ids[-1])+1,
                    frame_ids=ids, timestamps_seconds=ids/fps, extraction_id=np.asarray(extraction_id),
                    context_frame_start=np.asarray(int(ids[0])-core.start),
                    context_frame_stop=np.asarray(int(ids[0])-core.start+len(images)),
                    court_scores=np.asarray(court_scores_chunk),
                    shuttle_pixels=shuttle_result[:, :2], shuttle_detected=shuttle_result[:, 2].astype(bool),
                    pose_offsets=np.asarray(pose_offsets, dtype=np.int64),
                    boxes_xyxy=np.concatenate(boxes_chunk), box_scores=np.concatenate(box_scores_chunk),
                    keypoints_px=np.concatenate(points_chunk), keypoint_scores=np.concatenate(scores_chunk))
    finally:
        capture.release()

# Replay contains no video decoding or model calls. Raw arrays stay immutable.
all_ids, all_positions, all_radii, all_shuttle, all_valid = [], [], [], [], []
all_court_scores = []
for observations_chunk in raw_store.iter_chunks():
    positions_chunk, radii_chunk, valid_chunk = positions_from_observations(
        observations_chunk, reference_H, court_min_score, foot_min_score, court_margin_m)
    all_ids.append(observations_chunk['frame_ids'])
    all_positions.append(positions_chunk)
    all_radii.append(radii_chunk)
    all_shuttle.append(observations_chunk['shuttle_pixels'])
    all_valid.append(valid_chunk)
    all_court_scores.append(observations_chunk['court_scores'])
frame_ids = np.concatenate(all_ids)
raw_positions = np.concatenate(all_positions)
raw_stance_radii = np.concatenate(all_radii)
shuttle_pixels = np.concatenate(all_shuttle)
eligible = np.concatenate(all_valid)
court_scores = np.concatenate(all_court_scores)

jump_settings, jump_arrays, jump_events = {}, {}, []
if (jump_checkpoint is None) != (jump_projection_path is None):
    raise ValueError('Set both jump_checkpoint and jump_projection_path to enable jump correction.')
if jump_checkpoint is not None:
    import torch
    from badminton_analysis.jumps import JumpSegmenter, collect_player_poses, predict_airborne, rectify_jumps

    jump_model = JumpSegmenter()
    jump_model.load_state_dict(torch.load(jump_checkpoint, map_location='cpu', weights_only=True))
    jump_sequence = collect_player_poses(raw_store, reference_H, court_min_score, foot_min_score, court_margin_m)
    jump_probabilities = predict_airborne(jump_model, jump_sequence)
    jump_projection = np.load(jump_projection_path, allow_pickle=False)
    raw_positions, jump_xyz, jump_events = rectify_jumps(
        jump_sequence, raw_positions, jump_probabilities, jump_projection, fps, jump_threshold)
    raw_stance_radii[jump_probabilities >= jump_threshold] = np.nan
    eligible = np.isfinite(raw_positions).all(axis=-1).any(axis=1)
    jump_arrays = {'jump_probabilities': jump_probabilities, 'jump_xyz_m': jump_xyz}
    jump_settings = {'jumps': {'weights': file_fingerprint(jump_checkpoint, content=True),
                               'projection': jump_projection.tolist(), 'threshold': jump_threshold}}
accepted = motion_gate(raw_positions, shuttle_pixels, eligible, motion_window,
                       shuttle_deviation_px, player_displacement_m)
positions = raw_positions.copy()
positions[~accepted] = np.nan
stance_radii = raw_stance_radii.copy()
stance_radii[~accepted] = np.nan
print(f'Extracted {len(missing_ranges)} missing chunks and replayed {len(frame_ids):,} frames in {(time.monotonic()-started)/60:.1f} minutes.')
print(f'Court + available-player checks: {eligible.sum():,}; motion accepted: {accepted.sum():,} ({accepted.mean():.1%}).')
court_pass = court_scores >= court_min_score
rejection_counts = {
    'court': int((~court_pass).sum()),
    'players': int((court_pass & ~eligible).sum()),
    'motion': int((eligible & ~accepted).sum()),
}
assert sum(rejection_counts.values()) + int(accepted.sum()) == len(frame_ids)
observed_per_player = np.isfinite(positions).all(axis=2).sum(axis=0)
print(f'Video duration: {len(frame_ids)/fps/60:.2f} min')
print('Heatmap coverage:', {name: {'frames': int(observed_per_player[slot]),
      'minutes': round(observed_per_player[slot]/fps/60, 2),
      'fraction': round(observed_per_player[slot]/len(frame_ids), 4)}
      for slot, name in enumerate(['near', 'far'])})
print('Excluded seconds by reason:', {reason: round(count/fps, 2)
                                     for reason, count in rejection_counts.items()})
print('Coverage measures retained video observations; it is not the full match duration.')

## Heatmaps and saved observations

The heatmap value at each grid cell is the number of accepted source frames whose estimated player position falls within **15 cm** of that cell, divided by FPS. This is local dwell time, not a probability distribution; overlapping circles mean summing heatmap cells is not total duration. Grid cells are approximately 5 cm wide. Raw unsmoothed bin counts are also retained.

The plot includes the court margin so positions outside the boundary remain visible. Both players share the same seconds scale. CSV rows retain every source frame and both players; missing positions have zero dwell time. The heatmap NPZ retains raw bins, circular-presence frame counts, seconds and grid edges. `analysis.npz` also retains pre-motion positions and stance radii, eligibility, accepted flags and shuttle positions. These pre-motion positions are **derived**, distinct from raw image-space model poses.

Derived files live under this raw run's `derived/<analysis-id>/`; the analysis ID includes all filtering/heatmap settings and homography. Threshold experiments keep separate results. Repeating identical settings replaces only that derived result. All files are local persistent storage, excluded from Git; back up `data/player-position-heatmap/` if needed. Existing `.cache/player-position-heatmap/` exports predate raw capture and cannot restore discarded poses or shuttle detections.

For other experiments, open an existing run with `store = ObservationStore.open(run_directory)`, then use `store.load_calibration()` and `store.iter_chunks()` to read numeric arrays directly. This also works without the original video or model weights. Each chunk's `frame_ids` indexes its per-frame arrays; `pose_offsets` maps each frame ID to its poses. Calibration records its reference frame ID and applicable source-frame interval. Changing homography is possible for reprojection, but court support scores belong to the saved calibration and must be rescored from video for a new view.

New raw chunks contain a scalar `extraction_id` pointing to `extractions/<id>.json` for their small provenance record. Legacy chunks without this field inherit the original `manifest.json`. Devices can therefore vary across chunks without relabeling old observations; all poses, shuttle observations and frame IDs remain binary arrays.

In [ ]:
heatmaps = position_heatmaps(positions, fps, heatmap_radius_m, heatmap_cell_m, court_margin_m)
rows = []
for i, frame_id in enumerate(frame_ids):
    for slot, player in enumerate(['near', 'far']):
        rows.append({'frame': int(frame_id), 'player': player,
                     'x_m': positions[i, slot, 0], 'y_m': positions[i, slot, 1],
                     'stance_radius_m': stance_radii[i, slot],
                     'dwell_seconds': float(np.isfinite(positions[i, slot]).all())/fps})
observations = pd.DataFrame(rows)
settings = {
    'raw_run_id': raw_store.run_id, 'raw_manifest': str(raw_store.directory / 'manifest.json'),
    'video': str(video_path), 'fps': fps, 'source_frame_start': first_frame_id,
    'source_frame_stop': stop_frame_id, 'processed_frames': len(frame_ids),
    'rejected_frames_by_reason': rejection_counts,
    'accepted_frames_per_player': dict(zip(['near', 'far'], map(int, observed_per_player))),
    'processing_width': processing_width,
    'chunk_frames': chunk_frames, 'court_min_score': court_min_score,
    'court_check': 'fixed-reference line support', 'court_max_lines': court_max_lines,
    'foot_min_score': foot_min_score, 'court_margin_m': court_margin_m,
    'position_policy': 'partial-feet-v3',
    'motion_policy': 'offline-window-coverage-v2',
    'motion_window': motion_window, 'shuttle_deviation_px': shuttle_deviation_px,
    'player_displacement_m': player_displacement_m, 'heatmap_radius_m': heatmap_radius_m,
    'heatmap_cell_m': heatmap_cell_m, 'reference_H_court_to_image': reference_H.tolist(),
    **jump_settings,
}
analysis_id = settings_id({key: value for key, value in settings.items()
                           if key != 'rejected_frames_by_reason'})
output_dir = raw_store.directory / 'derived' / analysis_id
output_dir.mkdir(parents=True, exist_ok=True)
observations.to_csv(output_dir / 'positions.csv', index=False)
np.savez_compressed(output_dir / 'heatmaps.npz', **heatmaps)
np.savez_compressed(output_dir / 'analysis.npz', frame_ids=frame_ids,
                    raw_positions_m=raw_positions, raw_stance_radii_m=raw_stance_radii,
                    positions_m=positions, stance_radii_m=stance_radii,
                    eligible=eligible, accepted=accepted, court_scores=court_scores,
                    shuttle_pixels=shuttle_pixels, **jump_arrays)
if jump_settings:
    (output_dir / 'jump-events.json').write_text(json.dumps(jump_events, indent=2)+'\n')
(output_dir / 'settings.json').write_text(json.dumps(settings, indent=2)+'\n')

fig, axes = plt.subplots(1, 2, figsize=(11, 10), constrained_layout=True)
extent = [heatmaps['x_edges_m'][0], heatmaps['x_edges_m'][-1],
          heatmaps['y_edges_m'][0], heatmaps['y_edges_m'][-1]]
vmax = max(float(heatmaps['dwell_seconds'].max()), 1/fps)
for slot, axis in enumerate(axes):
    image = axis.imshow(heatmaps['dwell_seconds'][slot], extent=extent, origin='lower',
                        cmap='magma', vmin=0, vmax=vmax, interpolation='nearest')
    for segment in court_model_segments:
        axis.plot(*segment.T, color='cyan', linewidth=.6)
    axis.axhline(court_length_m/2, color='white', linestyle='--', linewidth=.8)
    axis.set(xlabel='Court X (m)', ylabel='Court Y (m)',
             title=f"{['Near', 'Far'][slot]} player — {observed_per_player[slot]/fps:.1f} s observed")
    axis.set_aspect('equal')
fig.colorbar(image, ax=axes, label=f'Dwell seconds within {heatmap_radius_m*100:g} cm')
fig.savefig(output_dir / 'heatmaps.png', dpi=160)
plt.show()
if not accepted.any():
    print('No frames passed all checks. Inspect the calibration and thresholds before interpreting an empty map.')
print('Saved:', output_dir)
display(observations.head(10))

In [ ]:
# Optional side-by-side video: source footage and cached 2D court positions.
# The full timeline includes rejected frames, with filter reasons and provisional markers.
render_replay_video = False
replay_path = output_dir / 'positions-replay.mp4'
if render_replay_video:
    from badminton_analysis.replay import render_position_replay
    replay_path = render_position_replay(raw_store.directory, output_dir,
                                         encoder='h264_nvenc')
if replay_path.is_file():
    print('Video replay:', replay_path)

# Split the existing annotated replay using the saved acceptance flags.
split_replay_videos = False
if split_replay_videos:
    from badminton_analysis.replay import split_position_replay
    split_position_replay(output_dir, encoder='h264_nvenc')
# Focus on player/motion failures within an accepted court view.
export_court_passed_rejections = False
if export_court_passed_rejections:
    from badminton_analysis.replay import split_position_replay
    split_position_replay(output_dir, encoder='h264_nvenc',
                          rejected_court_passed_only=True)
for name in ('accepted-replay.mp4', 'rejected-replay.mp4', 'rejected-court-passed-replay.mp4'):
    if (output_dir / name).is_file():
        print(name, output_dir / name)
